# Task 3 — Rigorous Evaluation Pipeline for SGS Closure Models
**Owner: Vempadapu Shyamal Deepak** · EPITA DSA 2026

Builds the shared evaluation harness used by Tasks 1 and 2.  All metrics are
validated on synthetic tensors with known ground truth before being applied to
real model predictions.

| Component | Description |
|-----------|-------------|
| V1 | Standard a-priori metrics (Pearson, dissipation, alignment) |
| V2 | Invariant-based distribution metrics (JSD on I₁, I₂, I₃) |
| V3 | Regime-partitioned evaluation (Q-criterion low/medium/high) |
| Deeper | Failure case visualisation — worst 10 predictions per model |

In [ ]:
# Colab setup
import subprocess, os, sys, shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# Copy repo from Drive (repo is private — Prerona must have run the sync cell first)
DRIVE_REPO = Path("/content/drive/MyDrive/sgs_runs/SHEAR_repo")
if not Path("/content/SHEAR").exists():
    shutil.copytree(str(DRIVE_REPO), "/content/SHEAR")
    print("Repo copied from Drive")
else:
    print("Repo already present")

# Install dependencies
subprocess.run(["pip", "install", "-q", "torch-geometric", "scipy", "e3nn"], check=True)

# Set paths
REPO = Path("/content/SHEAR")
SRC  = REPO / "src"
sys.path.insert(0, str(SRC))
os.chdir("/content/SHEAR/notebooks")

# Copy Task 2 results from Drive
DRIVE_RESULTS = Path("/content/drive/MyDrive/sgs_runs/task2_results")
(REPO / "results").mkdir(exist_ok=True)
if DRIVE_RESULTS.exists():
    for f in DRIVE_RESULTS.iterdir():
        shutil.copy(f, REPO / "results" / f.name)
        print(f"Loaded: {f.name}")
else:
    print("WARNING: Drive results not found — run sudip_task2.ipynb and sync cell first")

# Verify
expected = ["smagorinsky_test_preds.npy", "wale_test_preds.npy",
            "beck_mlp_test_preds.npy", "van_gastelen_mlp_test_preds.npy"]
missing  = [f for f in expected if not (REPO / "results" / f).exists()]
print("Missing:", missing if missing else "None — ready to run!")


## 0. Setup

In [ ]:
# CPU only — no GPU needed for this notebook
import sys, os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = Path('.').resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DEVICE = torch.device('cpu')
RESULTS_CSV = REPO / 'results' / 'baselines_results.csv'
FIGURES_DIR = REPO / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Task 3: Evaluation Pipeline')
print(f'Repo root: {REPO}')

## 1. Metric Validation on Synthetic Tensors

Every metric is tested against known ground truth before being used on real data.
The notebook **fails loudly** if any validation fails.

In [ ]:
from evaluation.metrics import (
    pearson_per_component, jsd_invariants, dissipation_error,
    principal_axis_alignment, backscatter_fraction, compute_all_metrics_np,
)
from evaluation.regime import q_criterion, regime_partition

failures = []

# --- 1. Pearson: identical → r=1, orthogonal → r≈0 ---
rng = np.random.default_rng(99)
N = 2000
tau_a = torch.from_numpy(rng.standard_normal((N, 3, 3)).astype(np.float32))
# Make symmetric
tau_a = (tau_a + tau_a.transpose(-1, -2)) / 2

r_identical = pearson_per_component(tau_a, tau_a)['mean']
ok = abs(r_identical - 1.0) < 1e-4
print(f'Pearson(identical) = {r_identical:.6f}  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('Pearson identical')

# Uncorrelated pair
tau_b = torch.from_numpy(rng.standard_normal((N, 3, 3)).astype(np.float32))
tau_b = (tau_b + tau_b.transpose(-1, -2)) / 2
r_uncorr = pearson_per_component(tau_a, tau_b)['mean']
ok = abs(r_uncorr) < 0.10
print(f'Pearson(uncorrelated) = {r_uncorr:.4f}  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('Pearson uncorrelated')

# --- 2. JSD: identical distributions → JSD=0 ---
jsd_same = jsd_invariants(tau_a, tau_a)['mean']
ok = jsd_same < 0.01
print(f'JSD(identical) = {jsd_same:.6f}  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('JSD identical')

# --- 3. Dissipation: zero gradient → zero dissipation ---
grad_zero = torch.zeros(N, 3, 3)
diss_zero = dissipation_error(tau_a, tau_a, grad_zero)
ok = abs(diss_zero['mean_pred']) < 1e-6
print(f'Diss(zero grad) mean={diss_zero["mean_pred"]:.2e}  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('Dissipation zero')

# --- 4. Principal-axis alignment: same tensor → 0 degrees ---
align_same = principal_axis_alignment(tau_a, tau_a)['mean_deg']
ok = align_same < 1.0
print(f'Alignment(identical) = {align_same:.4f} deg  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('Alignment identical')

# --- 5. Backscatter: Smagorinsky-type prediction → 0 backscatter ---
grad_syn = torch.from_numpy(rng.standard_normal((N, 3, 3)).astype(np.float32))
S_syn = 0.5 * (grad_syn + grad_syn.transpose(-1, -2))
S_mag_syn = torch.sqrt(2.0 * (S_syn**2).sum(dim=(-2, -1), keepdim=True).sum(dim=-1))
tau_smag_syn = -2 * 0.17**2 * S_mag_syn.unsqueeze(-1) * S_syn
bs = backscatter_fraction(tau_smag_syn, grad_syn)
ok = bs < 1e-6
print(f'Backscatter(Smagorinsky) = {bs:.6f}  [{"PASS" if ok else "FAIL"}]')
if not ok: failures.append('Backscatter Smagorinsky')

# --- 6. Q-criterion: uniform gradient → Q should be finite ---
grad_uni = torch.ones(10, 3, 3)
Q = q_criterion(grad_uni)
ok = Q.isfinite().all()
print(f'Q-criterion finite: {ok}  [PASS]')

# --- Final ---
if failures:
    raise AssertionError(f'METRIC VALIDATION FAILED: {failures}')
else:
    print('\nAll metric validations PASSED.')

## 2. Load Test Data and All Model Predictions

In [ ]:
from data.jhtdb import generate_synthetic_les_data
from data.dataset import build_dataloaders

tau_field, grad_field = generate_synthetic_les_data(n_les=64, seed=0)
_, _, test_loader, _ = build_dataloaders(
    tau_field, grad_field,
    n_train=4000, n_val=500, n_test=500,
    patch_size=8, k_neighbours=26,
    batch_size=256, num_workers=0, seed=42,
)

grad_list, tau_list = [], []
for batch in test_loader:
    grad_list.append(batch.grad_full.numpy().reshape(-1, 3, 3))
    tau_list.append(batch.tau_full.numpy().reshape(-1, 3, 3))
grad_test = np.concatenate(grad_list, axis=0)
tau_test  = np.concatenate(tau_list,  axis=0)
grad_test_t = torch.from_numpy(grad_test.astype(np.float32))
tau_test_t  = torch.from_numpy(tau_test.astype(np.float32))

print(f'Test set: {grad_test.shape[0]:,} nodes from 500 sub-cubes')

# Load predictions saved by Tasks 1 and 2
pred_files = {
    'smagorinsky':      REPO / 'results' / 'smagorinsky_test_preds.npy',
    'wale':             REPO / 'results' / 'wale_test_preds.npy',
    'beck_mlp':         REPO / 'results' / 'beck_mlp_test_preds.npy',
    'van_gastelen_mlp': REPO / 'results' / 'van_gastelen_mlp_test_preds.npy',
}
predictions = {}
for name, path in pred_files.items():
    if path.exists():
        predictions[name] = torch.from_numpy(np.load(str(path)).astype(np.float32))
        print(f'  Loaded {name}: {predictions[name].shape}')
    else:
        print(f'  [{name}] not found — run Task 2 notebook first')

## 3. V1 — Standard A-Priori Metrics for All Models

In [ ]:
import time
from evaluation.metrics import compute_all_metrics_np, save_results_csv

all_metrics = {}
for name, pred_t in predictions.items():
    t0 = time.time()
    metrics = compute_all_metrics_np(pred_t, tau_test_t, grad_test_t)
    elapsed = time.time() - t0
    metrics['runtime_seconds'] = elapsed
    all_metrics[name] = metrics
    save_results_csv(metrics, name + '_task3', RESULTS_CSV)
    print(f'{name:<25}  Pearson={metrics["pearson"]["mean"]:.4f}  '
          f'JSD={metrics["jsd_invariants"]["mean"]:.4f}  '
          f'BS={metrics["backscatter_fraction"]:.4f}  '
          f't={elapsed:.1f}s')

## 4. V2 — Invariant-Based Distribution Metrics

In [ ]:
from evaluation.visualize import plot_invariant_distributions

for name, pred_t in predictions.items():
    save_path = str(FIGURES_DIR / f'{name}_invariant_distributions.png')
    plot_invariant_distributions(pred_t.numpy(), tau_test_t.numpy(), save_path=save_path)
    print(f'Saved invariant distribution plot: {save_path}')
    I1_jsd = all_metrics[name]['jsd_invariants']['I1']
    I2_jsd = all_metrics[name]['jsd_invariants']['I2']
    I3_jsd = all_metrics[name]['jsd_invariants']['I3']
    print(f'  {name}: JSD I1={I1_jsd:.4f}  I2={I2_jsd:.4f}  I3={I3_jsd:.4f}')

## 5. V3 — Regime-Partitioned Evaluation (Q-criterion)

In [ ]:
from evaluation.regime import metrics_by_regime, REGIME_NAMES
from evaluation.metrics import compute_all_metrics_np

regime_results = {}
for name, pred_t in predictions.items():
    regime_results[name] = metrics_by_regime(
        pred_t, tau_test_t, grad_test_t,
        compute_fn=compute_all_metrics_np,
    )

# Print Pearson r per regime
print(f'{"Model":<25} {"Low Q":>10} {"Medium Q":>10} {"High Q":>10} {"All":>10}')
print('-' * 60)
for name, rr in regime_results.items():
    row = f'{name:<25}'
    for regime in ['low', 'medium', 'high', 'all']:
        val = rr.get(regime, {}).get('pearson', {}).get('mean', float('nan'))
        row += f'{val:>10.4f}'
    print(row)

In [ ]:
# Plot regime Pearson r as grouped bar chart
regimes = ['low', 'medium', 'high']
model_names = list(regime_results.keys())
n_models = len(model_names)
x = np.arange(3)
width = 0.8 / n_models

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(model_names):
    vals = [regime_results[name].get(r, {}).get('pearson', {}).get('mean', np.nan)
            for r in regimes]
    ax.bar(x + i * width - (n_models-1)*width/2, vals, width, label=name, alpha=0.8)

ax.set_xticks(x); ax.set_xticklabels(['Low Q\n(straining)', 'Medium Q', 'High Q\n(vortex)'])
ax.set_ylabel('Pearson r (mean over τ components)')
ax.set_title('Closure model accuracy by turbulence intensity regime')
ax.legend(fontsize=8, loc='upper right'); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'regime_pearson.png'), dpi=150)
plt.show()

## 6. Failure Case Visualisation

In [ ]:
from evaluation.visualize import worst_predictions, plot_tau_slices, plot_q_overlay

for name, pred_t in predictions.items():
    worst_idx = worst_predictions(pred_t.numpy(), tau_test_t.numpy(), n=10)
    print(f'{name}: worst 10 sub-cubes = {worst_idx}')

    fail_dir = FIGURES_DIR / f'{name}_failures'
    fail_dir.mkdir(exist_ok=True)

    # τ_xy mid-plane slices
    plot_tau_slices(
        pred_t.numpy(), tau_test_t.numpy(),
        indices=worst_idx[:5].tolist(), component='tau_xy',
        save_path=str(fail_dir / 'tau_xy_slices.png'),
    )

    # τ_xy with Q-criterion overlay
    plot_q_overlay(
        pred_t.numpy(), tau_test_t.numpy(), grad_test,
        indices=worst_idx[:3].tolist(), component='tau_xy',
        save_path=str(fail_dir / 'tau_xy_q_overlay.png'),
    )
    print(f'  Figures saved to {fail_dir}')

### Physical interpretation

If the 10 worst-predicted sub-cubes cluster in the **high-Q regime** (strong vortices),
this confirms that the Boussinesq (eddy-viscosity) assumption breaks down exactly where
we expect — directly motivating the equivariant generative closure approach of Task 1.

In [ ]:
from evaluation.regime import q_criterion, per_sample_q

Q_per_sample = per_sample_q(grad_test_t, n_nodes_per_sample=512)  # [500]
Q_np = Q_per_sample.numpy()

for name, pred_t in predictions.items():
    worst_idx = worst_predictions(pred_t.numpy(), tau_test_t.numpy(), n=10)
    q_worst = Q_np[worst_idx]
    q_all   = Q_np
    from scipy.stats import percentileofscore
    pct = percentileofscore(q_all, q_worst.mean())
    print(f'{name:<25}: worst-10 mean Q at {pct:.0f}th percentile  '
          f'({"high-Q regime" if pct > 66 else "medium" if pct > 33 else "low-Q regime"})')

## 7. Correlation Scatter Plots

In [ ]:
from evaluation.visualize import plot_correlation_scatter

for name, pred_t in predictions.items():
    plot_correlation_scatter(
        pred_t.numpy(), tau_test_t.numpy(),
        component='tau_xy',
        save_path=str(FIGURES_DIR / f'{name}_scatter_tau_xy.png'),
    )
print('Scatter plots saved.')

## 8. Inference Runtime Report

In [ ]:
print('Inference runtimes (CPU):')
for name, m in all_metrics.items():
    t = m.get('runtime_seconds', float('nan'))
    n_nodes = grad_test.shape[0]
    ms_per_cube = 1000 * t / 500 if t == t else float('nan')
    print(f'  {name:<25}: total={t:.2f}s  ~{ms_per_cube:.2f} ms/sub-cube')

## 9. Final Summary Table (Paper-ready)

In [ ]:
from evaluation.metrics import load_results_csv

results_all = load_results_csv(RESULTS_CSV)

print('\n' + '='*100)
print('FINAL RESULTS TABLE — Task 3 Evaluation Pipeline')
print('='*100)
metrics_table = [
    ('pearson.mean',             'Pearson r'),
    ('jsd_invariants.mean',      'JSD (mean)'),
    ('dissipation.correlation',  'Diss corr'),
    ('alignment.mean_deg',       'Align (deg)'),
    ('backscatter_fraction',     'Backscatter'),
]
header = f'{"Model":<28}' + ''.join(f'{lbl:<15}' for _, lbl in metrics_table)
print(header)
print('-' * 100)
for model_name, m in sorted(results_all.items()):
    row = f'{model_name:<28}'
    for key, _ in metrics_table:
        val = m.get(key, float('nan'))
        row += f'{val:<15.4f}' if isinstance(val, float) else f'{str(val):<15}'
    print(row)
print('='*100)